In [27]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[0] / "src"))

In [29]:
from datetime import datetime
from pathlib import Path

from viberank.datasets.hmls_rc_dataloader import RankCentralityDataLoader
from viberank.experiments.rank_centrality import RankCentralityExperimentRunner
from viberank.utils.logging import JSONLResponseLogger
from viberank.comparators.dummy import DummyComparator
from viberank.comparators.LLMComparatorFullData import LLMComparator

from viberank.datasets.hmls_dataloader_all import HMISPairwiseDataLoader


In [30]:
config_path = Path("../configs/datasets/rc_full_data.yaml")



In [ ]:
dataloader = HMISPairwiseDataLoader.from_yaml(config_path)
dataloader.prepare()

run_id = datetime.now().strftime("AIES_vispdat_%Y%m%d_%H%M%S")
log_path = dataloader.config.responses_dir / f"{run_id}.jsonl"

logger = JSONLResponseLogger(
    log_path=log_path,
    flush_every=1,
    store_prompts=True,
)

[HMISPairwiseDataLoader] ================================================================================
[HMISPairwiseDataLoader] Starting dataloader.prepare()
[HMISPairwiseDataLoader] dataset_name: VIFSPDAT
[HMISPairwiseDataLoader] household_source: all
[HMISPairwiseDataLoader] raw_root: C:\Users\ShafkatFarabi\Box\WashU-ICA-SFT-Working Folder - shared\Sandboxes\Gaurab Pokharel\VibeRank\raw\hmls
[HMISPairwiseDataLoader] processed_root: C:\Users\ShafkatFarabi\Box\WashU-ICA-SFT-Working Folder - shared\Sandboxes\Gaurab Pokharel\VibeRank\processed\rank_centrality\hmls
[HMISPairwiseDataLoader] responses_dir: C:\Users\ShafkatFarabi\Box\WashU-ICA-SFT-Working Folder - shared\Sandboxes\Gaurab Pokharel\VibeRank\raw\hmls\VIFSPDAT\rc_responses
[HMISPairwiseDataLoader] ================================================================================
[HMISPairwiseDataLoader] Loading ALL households from raw JSON folders
[HMISPairwiseDataLoader] raw_dataset_dir: C:\Users\ShafkatFarabi\Box\WashU-ICA-SF

In [32]:
len(dataloader.tie_sheet)

97301

In [33]:

comp_kwargs = dataloader.get_comparator_kwargs()

In [34]:
comp_kwargs["prompt_path"] = "/projects/simlai1/Viberank/data/raw/hmls/prompt_vulnerability.txt"

In [ ]:
pairs = comp_kwargs.pop("pairs", None)
comp = LLMComparator(
    **comp_kwargs,
    num_samples=dataloader.config.run_settings.get("repeats_per_ordered_pair", 10),
    logger = logger,
    rng_seed=42, 
    llm_name = 'qwen', # deepseek8B / llama7 / qwen
    timeout= 120,
    max_tokens = 256,
    temperature = 0.1,
    local_test_mode = False
    #prompt_path='prompt_vulnerability.txt'
)

initialized LLMComparator in LOCAL TEST MODE: no vLLM/model loaded


In [ ]:
runner = RankCentralityExperimentRunner(
    dataloader=dataloader,
    logger=logger,
    comparator=comp,
    run_id="aies_vifspdat_qwen_001",
    model_name="qwen",
    prompt_version="v1",
)

In [38]:
result = runner.run()

[HMISPairwiseDataLoader] ================================================================================
[HMISPairwiseDataLoader] Starting dataloader.prepare()
[HMISPairwiseDataLoader] dataset_name: VIFSPDAT
[HMISPairwiseDataLoader] household_source: all
[HMISPairwiseDataLoader] raw_root: C:\Users\ShafkatFarabi\Box\WashU-ICA-SFT-Working Folder - shared\Sandboxes\Gaurab Pokharel\VibeRank\raw\hmls
[HMISPairwiseDataLoader] processed_root: C:\Users\ShafkatFarabi\Box\WashU-ICA-SFT-Working Folder - shared\Sandboxes\Gaurab Pokharel\VibeRank\processed\rank_centrality\hmls
[HMISPairwiseDataLoader] responses_dir: C:\Users\ShafkatFarabi\Box\WashU-ICA-SFT-Working Folder - shared\Sandboxes\Gaurab Pokharel\VibeRank\raw\hmls\VIFSPDAT\rc_responses
[HMISPairwiseDataLoader] ================================================================================
[HMISPairwiseDataLoader] Loading ALL households from raw JSON folders
[HMISPairwiseDataLoader] raw_dataset_dir: C:\Users\ShafkatFarabi\Box\WashU-ICA-SF

KeyboardInterrupt: 

In [ ]:
print(result)